# Road Inspection — train the India road-damage model on Kaggle's free GPU

**Before running**
1. *Settings → Accelerator*: **GPU T4 x2** (or P100). *Settings → Internet*: **On**
   (needs a phone-verified Kaggle account).
2. *Add Input*: your dataset made from `road-inspection-india.zip`.
3. Run it as a background job: **Save Version → Save & Run All (Commit)**.
   The commit keeps running after you close the browser (12 h session limit) and
   saves everything in `/kaggle/working` as the notebook's output.

When it finishes, open the version's **Output** tab and download `india_model.zip`.

In [ ]:
# Settings — change PROFILE to "best" for the larger yolo11m model (needs ~2.5x the time)
PROFILE = "laptop"          # yolo11s @ 768 px
HOURS = 10.5                # training budget; leaves time for setup + evaluation in a 12 h session
RUN_NAME = f"india_{PROFILE}"

In [ ]:
import subprocess, torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — enable it in Settings")
!pip install -q "ultralytics>=8.3" pyyaml

In [ ]:
# Find the uploaded dataset (Kaggle unzips it somewhere under /kaggle/input)
from pathlib import Path
roads = sorted(Path("/kaggle/input").rglob("road_v2"))
assert roads, "road_v2 not found — did you add the dataset as input?"
ROAD = roads[0]
BUNDLE = ROAD.parent
SCRIPTS = BUNDLE / "scripts" / "training"
BASELINE = BUNDLE / "artifacts" / "ai_models" / "best.pt"
WORK = Path("/kaggle/working")
print("data:", ROAD, "| scripts:", SCRIPTS.exists(), "| baseline model:", BASELINE.exists())
for split in ("train", "val", "test"):
    print(split, len(list((ROAD / split / "images").iterdir())), "images")

In [ ]:
# 1. Build the balanced, de-duplicated training set
!python {SCRIPTS}/build_dataset.py --road {ROAD} --extra /kaggle/working/no_extra --out {WORK}/build/india_v3

In [ ]:
# 2. Train (AMP on: it is stable on Kaggle GPUs; the collapse guard stops the run if anything goes NaN)
!python {SCRIPTS}/train.py --data {WORK}/build/india_v3/data.yaml --profile {PROFILE} --device 0 --workers 4 --amp --hours {HOURS} --project {WORK}/runs --name {RUN_NAME}

In [ ]:
# 3. Evaluate against the current production model on India val/test (report only — promotion happens on your PC)
BEST = WORK / "runs" / RUN_NAME / "weights" / "best.pt"
assert BEST.exists(), "training did not produce best.pt — check the training cell output"
baseline_arg = f"--baseline {BASELINE}" if BASELINE.exists() else f"--baseline {BEST}"
!python {SCRIPTS}/evaluate.py --candidate {BEST} {baseline_arg} --data {WORK}/build/india_v3/data.yaml --device 0 || true

In [ ]:
# 4. Pack the results for download (best.pt + metrics, curves, evaluation report)
import zipfile
run = WORK / "runs" / RUN_NAME
out = WORK / "india_model.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in run.rglob("*"):
        if f.is_file() and f.name != "last.pt":
            zf.write(f, f"{RUN_NAME}/{f.relative_to(run)}")
print(out, round(out.stat().st_size / 1e6, 1), "MB")
print((run / "evaluation" / "evaluation.md").read_text() if (run / "evaluation" / "evaluation.md").exists() else "no evaluation report")

## Back on your PC (project root)

1. Unzip `india_model.zip` into `artifacts/runs/` (you get `artifacts/runs/india_laptop/...`).
2. Re-run the gate locally and promote the model into the app:

```
python scripts/training/evaluate.py --candidate artifacts/runs/india_laptop/weights/best.pt --device cpu --promote
```

3. Restart the app (`start.bat`). It now uses the new model and its per-class thresholds.